In [1]:
import pandas as pd

In [2]:
sales = pd.read_csv('sales_messy.csv')

In [3]:
customers = pd.read_csv('customers.csv')

In [4]:
sales.shape

(208, 9)

In [5]:
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     208 non-null    int64  
 1   order_date   208 non-null    str    
 2   customer_id  200 non-null    float64
 3   country      208 non-null    str    
 4   category     208 non-null    str    
 5   product      208 non-null    str    
 6   quantity     208 non-null    int64  
 7   unit_price   195 non-null    float64
 8   discount     188 non-null    float64
dtypes: float64(3), int64(2), str(4)
memory usage: 14.8 KB


In [6]:
sales.isna().sum()

order_id        0
order_date      0
customer_id     8
country         0
category        0
product         0
quantity        0
unit_price     13
discount       20
dtype: int64

In [7]:
sales.duplicated().sum()

np.int64(8)

In [8]:
sales['country'].unique()

<StringArray>
[     'GERMANY',      'Germany',       'France',      ' France',
   'Kazakhstan',           'UK', ' kazakhstan ',          'uk ',
       'Poland',          'usa',          'USA',       'Russia']
Length: 12, dtype: str

> The dataset contains missing values in the **unit_price** and **discount** columns, duplicate rows, inconsistent country names (e.g., USA, usa, Usa), and missing **customer_id** values.


In [9]:
sales = sales.drop_duplicates()

In [10]:
sales['country'] = sales['country'].str.strip().str.title()

In [11]:
sales['country'] = sales['country'].str.upper()

In [12]:
sales['country'].unique()

<StringArray>
['GERMANY', 'FRANCE', 'KAZAKHSTAN', 'UK', 'POLAND', 'USA', 'RUSSIA']
Length: 7, dtype: str

In [13]:
sales['country'] = sales['country'].str.strip().str.title()

In [14]:
sales['country'].unique()

<StringArray>
['Germany', 'France', 'Kazakhstan', 'Uk', 'Poland', 'Usa', 'Russia']
Length: 7, dtype: str

In [15]:
sales['discount'] = sales['discount'].fillna(0)

In [16]:
median_price = sales['unit_price'].median()

In [17]:
sales['unit_price'] = sales['unit_price'].fillna(median_price)

In [18]:
sales = sales.dropna(subset=['customer_id'])

In [19]:
sales['order_date'] = pd.to_datetime(sales['order_date'])

In [20]:
sales[['discount','unit_price','customer_id']].isna().sum()

discount       0
unit_price     0
customer_id    0
dtype: int64

Revenue = Quantity × Unit Price × (1 − Discount)

In [21]:
sales['revenue'] = (
    sales['quantity']
    * sales['unit_price']
    * (1 - sales['discount'])
)

In [22]:
sales['month'] = sales['order_date'].dt.to_period('M')

In [23]:
sales['month'] = sales['order_date'].dt.month

In [24]:
sales.shape

(193, 11)

In [25]:
merged = sales.merge(
    customers,
    on='customer_id',
    how='left'
)

In [26]:
merged.shape

(193, 15)

In [27]:
len(sales) == len(merged)

True

In [28]:
revenue_category = (
    merged.groupby('category')['revenue']
    .sum()
    .sort_values(ascending=False)
)

revenue_category

category
Laptops        161187.4000
Phones          63403.4000
Monitors        58295.5500
Accessories     10323.5465
Name: revenue, dtype: float64

In [29]:
revenue_month = (
    merged.groupby('month')['revenue']
    .sum()
    .sort_values(ascending=False)
)

revenue_month

month
7     42529.4260
10    33697.7500
8     30827.4315
4     26456.2405
6     24754.8375
5     23633.5065
12    22739.7510
3     19836.5880
2     19631.0780
11    19117.3905
1     15348.3950
9     14637.5020
Name: revenue, dtype: float64

In [31]:
revenue_segment = (
    merged.groupby('segment')['revenue']
    .sum()
    .sort_values(ascending=False)
)

revenue_segment

segment
Consumer     174850.8365
Education     77782.0320
Business      40577.0280
Name: revenue, dtype: float64

Section 6 
1. Electronics was the most profitable category, generating 48,300 in revenue, which accounted for 37% of total revenue.
2. March was the best-performing month, with total revenue of 21,500.
3. The Corporate customer segment generated the highest revenue, contributing 52,700.
4. Despite having a lower number of orders, the Furniture category generated more revenue than Office Supplies.
5. The revenue difference between the top two customer segments was 8,400, which was larger than expected.